In [20]:
from google import genai
from google.genai import types
from PIL import Image

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
# 1. Initialize the client (automatically picks up GEMINI_API_KEY from environment)
client = genai.Client(
    vertexai=True,
    project="infinitas-ds-ai-poc",
    location="us-central1",
)

In [22]:
# 2. Open your local image file

path_batch1 = ["test_images/First_Tamper.png", "test_images/first-tamper.png", "test_images/pink.png","test_images/combined.png",
         "test_images/neat.png", "test_images/neatest.png", "test_images/cropped-obvious.png"] # explore strength and weaknesses

path_batch2 = ["test_images/payslip1.jpeg", "test_images/payslip10.jpeg", "test_images/payslip2.png",
               "test_images/payslip20.png","test_images/payslip3.png", "test_images/payslip30.png"] # explore payslip with white space but different fonts

paths = path_batch1
images = []

for path in paths:
    images.append(Image.open(path))

print(len(images))

7


## Ground truth labels

Set the true `label` ("tampered" / "authentic") and `label_signals` (true signal types present, if tampered) for each image in this batch. Stored in `image_labels.json`, keyed by image path.

`log_result()` looks these up by `image_path` automatically when logging — they are for analysis only and are **never** included in the prompt/model call. Run this once per image (re-running just overwrites with the same value).

In [23]:
# # Only need to set up after making a new image once after it's been created.

# from result_logger import set_image_label, load_image_labels

# # Fill in the real ground truth for each image in `paths` below.
# # label: "tampered" or "authentic"
# # label_signals: list of signal types actually present (only meaningful if "tampered")
# #   -- see EXPECTED_SIGNAL_TYPES in result_logger.py for valid values

# ground_truth: dict[str, tuple[str, list[str]]] = {
#     paths[0]: ("authentic", []),
#     paths[1]: ("tampered", ["background_overlay", "font_weight_inconsistency",]),
#     paths[2]: ("tampered", ["background_overlay", "font_weight_inconsistency"]),
#     paths[3]: ("tampered", ["background_overlay", "font_weight_inconsistency"]),
#     paths[4]: ("tampered", ["font_weight_inconsistency","baseline_mismatch"]),
#     paths[5]: ("tampered", ["font_weight_inconsistency"]),
#     paths[6]: ("tampered", ["background_overlay", "font_weight_inconsistency", "baseline_mismatch"]),
# }

# for image_path, (label, label_signals) in ground_truth.items():
#     set_image_label(image_path, label, label_signals)

# load_image_labels()

In [24]:
# Read the V1 prompt

with open("prompt-library/V1.txt", "r", encoding="utf-8") as file:
    file_content = file.read()

prompt_v1 = file_content
print(type(prompt_v1))

# Read the V1 prompt split into system instruction + task prompt

with open("prompt-library/V1_system.txt", "r", encoding="utf-8") as file:
    system_prompt_v1 = file.read()

with open("prompt-library/V1_task.txt", "r", encoding="utf-8") as file:
    task_prompt_v1 = file.read()

<class 'str'>


## Result logging

Every model call below is logged to `notebook_results/results_log.jsonl` via `result_logger.log_result()`.
This keeps a permanent record across batches without cluttering this notebook.

**For each new batch of images:**
1. Bump `BATCH_ID` below (e.g. `"batch2"`)
2. Update `paths` in the image-loading cell to point at the new images
3. Re-run the experiment cells as usual — results from previous batches stay in the log

Use the "Review logged results" section at the end to compare across batches.

In [25]:
import time
from result_logger import log_result, load_results

BATCH_ID = "batch1"  # bump this for each new set of test images

## Try with the same model for now to test the workflow
### Models (to double check)
- "gemini-2.5-flash"
- "gemini-2.5-pro"
- "gemini-3.5-flash"
- "gemini-3.5-pro"

In [15]:
image_list = list(zip(paths, images))  

for image_path, image in image_list:

    start = time.perf_counter()

    # 3. Generate content by passing both the image object and your text prompt
    response = client.models.generate_content(
        model="gemini-2.5-pro",
        contents=[
            prompt_v1,
            image
        ], config = types.GenerateContentConfig(
            temperature = 0.1 # To change to variable this later
        )
    )

    latency_s = round(time.perf_counter() - start, 2)

    # 4. Print the text result
    print(response.text)

    log_result(
        batch_id=BATCH_ID,
        image_path=image_path,
        prompt_id="V1",
        model="gemini-2.5-pro",
        temperature=0.1,
        raw_response=response.text,
        latency_s=latency_s,
    )

```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document was scanned to establish a typographic baseline. All text appears to be rendered in the same font family, with consistent sharpness, weight, and resolution. The title is slightly larger, which is a standard formatting choice. The body text, including the address and the date, is uniform. There are no color or contrast discontinuities, no mismatched backgrounds, and no pixelation anomalies around any characters or numbers. The Thai numerals in the postal code and the date blend seamlessly with the surrounding text. Overall, there is no visual evidence to suggest any part of the text was digitally altered after the document's initial creation.",
  "signals_detected": [],
  "notes": "The document is a very clean, digitally-generated image. This high quality makes it easier to spot inconsistencies, and none were found. The 'authentic' verdict refers only to the absence of post-recapture digital tamperi

In [16]:
# V1 split: persona passed as system_instruction, task prompt as the user turn

image_list = list(zip(paths, images))  # image 1 and 2

for image_path, image in image_list:
    start = time.perf_counter()

    response = client.models.generate_content(
        model="gemini-2.5-pro",
        contents=[task_prompt_v1, image],
        config=types.GenerateContentConfig(
            system_instruction=system_prompt_v1,
            temperature = 0.1
        ),
    )

    latency_s = round(time.perf_counter() - start, 2)

    print(response.text)

    log_result(
        batch_id=BATCH_ID,
        image_path=image_path,
        prompt_id="V1_split",
        model="gemini-2.5-pro",
        temperature=0.1,
        raw_response=response.text,
        latency_s=latency_s,
    )

```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "I performed a thorough visual analysis of the document. The typography, including font type, weight, sharpness, and color, is consistent across all three lines of text. The background is a uniform white with no evidence of color or contrast discontinuities that would suggest text has been pasted or altered. The character edges are uniformly crisp, and there are no resolution mismatches between different words or numbers. The spacing and alignment of all text elements appear natural and consistent with a document created in a single session. No visual signals of post-recapture digital modification were found.",
  "signals_detected": [],
  "notes": "The image quality is extremely high, suggesting it is a direct digital export or a 'born-digital' document rather than a scan or photograph of a physical paper. This high level of clarity makes it easier to confirm the absence of tampering artifacts."
}
```
```json
{


## Review logged results

Load the full log (all batches) or just the current `BATCH_ID` to compare prompts/models without re-running anything.

In [26]:
df_all = load_results()
df_all[["timestamp", "batch_id", "image_path", "prompt_id", "model", "verdict", "confidence", "signal_types", "format", "latency_s"]]

,timestamp,batch_id,image_path,prompt_id,model,verdict,confidence,signal_types,format,latency_s
0,2026-06-11T02:20:36.332953+00:00,batch1,test_images/First_Tamper.png,V1,gemini-2.5-flash,authentic,high,[],True,9.42
1,2026-06-11T02:20:47.181659+00:00,batch1,test_images/first-tamper.png,V1,gemini-2.5-flash,tampered,medium,[resolution_mismatch],True,10.85
2,2026-06-11T02:20:53.951337+00:00,batch1,test_images/First_Tamper.png,V1_split,gemini-2.5-flash,authentic,high,[],True,6.76
3,2026-06-11T02:21:04.694144+00:00,batch1,test_images/first-tamper.png,V1_split,gemini-2.5-flash,authentic,high,[],True,10.74
4,2026-06-11T02:24:43.831825+00:00,batch1,test_images/First_Tamper.png,V1,gemini-2.5-flash,authentic,high,[],True,9.18
...,...,...,...,...,...,...,...,...,...,...
75,2026-06-11T07:51:36.241138+00:00,batch1,test_images/pink.png,V1_split,gemini-2.5-pro,tampered,high,"[color_contrast_discontinuity, font_weight_inc...",True,18.18
76,2026-06-11T07:51:57.597558+00:00,batch1,test_images/combined.png,V1_split,gemini-2.5-pro,tampered,high,"[color_contrast_discontinuity, color_contrast_...",True,21.35
77,2026-06-11T07:52:22.938434+00:00,batch1,test_images/neat.png,V1_split,gemini-2.5-pro,tampered,high,"[resolution_mismatch, color_contrast_discontin...",True,25.34
78,2026-06-11T07:52:46.607797+00:00,batch1,test_images/neatest.png,V1_split,gemini-2.5-pro,tampered,high,"[font_weight_inconsistency, resolution_mismatch]",True,23.67


In [27]:
df_all.columns

Index(['timestamp', 'batch_id', 'image_path', 'prompt_id', 'model',
       'temperature', 'latency_s', 'raw_response', 'parsed_response', 'notes',
       'label', 'label_signals', 'verdict', 'confidence', 'signal_types',
       'format'],
      dtype='object')

In [28]:
df_all[df_all['batch_id'] == 'batch1']

,timestamp,batch_id,image_path,prompt_id,model,temperature,latency_s,raw_response,parsed_response,notes,label,label_signals,verdict,confidence,signal_types,format
0,2026-06-11T02:20:36.332953+00:00,batch1,test_images/First_Tamper.png,V1,gemini-2.5-flash,0.10,9.42,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True
1,2026-06-11T02:20:47.181659+00:00,batch1,test_images/first-tamper.png,V1,gemini-2.5-flash,0.10,10.85,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'medium'...",,tampered,"[background_overlay, font_weight_inconsistency]",tampered,medium,[resolution_mismatch],True
2,2026-06-11T02:20:53.951337+00:00,batch1,test_images/First_Tamper.png,V1_split,gemini-2.5-flash,0.10,6.76,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True
3,2026-06-11T02:21:04.694144+00:00,batch1,test_images/first-tamper.png,V1_split,gemini-2.5-flash,0.10,10.74,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[background_overlay, font_weight_inconsistency]",authentic,high,[],True
4,2026-06-11T02:24:43.831825+00:00,batch1,test_images/First_Tamper.png,V1,gemini-2.5-flash,0.05,9.18,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True
5,2026-06-11T02:24:51.311197+00:00,batch1,test_images/first-tamper.png,V1,gemini-2.5-flash,0.05,7.48,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[background_overlay, font_weight_inconsistency]",authentic,high,[],True
6,2026-06-11T02:25:08.611948+00:00,batch1,test_images/combined.png,V1,gemini-2.5-flash,0.05,17.30,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,tampered,"[background_overlay, font_weight_inconsistency]",tampered,high,"[background_anomaly, background_anomaly]",False
7,2026-06-11T02:25:16.513412+00:00,batch1,test_images/neat.png,V1,gemini-2.5-flash,0.05,7.90,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[font_weight_inconsistency, baseline_mismatch]",authentic,high,[],True
8,2026-06-11T02:25:28.105398+00:00,batch1,test_images/neatest.png,V1,gemini-2.5-flash,0.05,11.59,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'medium'...",,tampered,[font_weight_inconsistency],tampered,medium,[font_weight_inconsistency],True
9,2026-06-11T02:25:42.508209+00:00,batch1,test_images/cropped-obvious.png,V1,gemini-2.5-flash,0.05,14.40,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[background_overlay, font_weight_inconsistency...",authentic,high,[],True


In [35]:
# Calculate some accuracy, recall, F1 metrics
total_tests = len(df_all)
total_wrong_verdict = len(df_all[df_all['label'] != df_all['verdict']])

print("Wrong_verdict_percent", total_wrong_verdict/total_tests*100)

Wrong_verdict_percent 35.0


### Different batches will be different I am sure

In [41]:
df_all['verdict'].value_counts()

verdict
tampered        40
authentic       39
inconclusive     1
Name: count, dtype: int64

In [39]:
df_all[(df_all['verdict'] == 'inconclusive')]

,timestamp,batch_id,image_path,prompt_id,model,temperature,latency_s,raw_response,parsed_response,notes,label,label_signals,verdict,confidence,signal_types,format
42,2026-06-11T06:06:48.188613+00:00,batch2,test_images/payslip1.jpeg,V1,gemini-2.5-flash,0.05,31.21,"```json\n{\n ""verdict"": ""inconclusive"",\n ""c...","{'verdict': 'inconclusive', 'confidence': 'hig...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",inconclusive,high,[font_weight_inconsistency],True


In [43]:
from analysis_util import compute_binary_metrics
total_matrics = compute_binary_metrics(df_all)
print(total_matrics)

precision          0.825000
recall             0.622642
specificity        0.730769
f1                 0.709677
accuracy           0.658228
true_positive     33.000000
false_positive     7.000000
false_negative    20.000000
true_negative     19.000000
excluded_count     1.000000
dtype: float64


#### Batch 1

In [46]:
batch1 = df_all[df_all['batch_id'] =='batch1']
print(compute_binary_metrics(batch1))

precision          1.000000
recall             0.638889
specificity        1.000000
f1                 0.779661
accuracy           0.704545
true_positive     23.000000
false_positive     0.000000
false_negative    13.000000
true_negative      8.000000
excluded_count     0.000000
dtype: float64


#### Batch 2

In [48]:
batch2 = df_all[df_all['batch_id'] =='batch2']
print(compute_binary_metrics(batch2))

precision          0.588235
recall             0.588235
specificity        0.611111
f1                 0.588235
accuracy           0.600000
true_positive     10.000000
false_positive     7.000000
false_negative     7.000000
true_negative     11.000000
excluded_count     1.000000
dtype: float64


In [49]:
batch2

,timestamp,batch_id,image_path,prompt_id,model,temperature,latency_s,raw_response,parsed_response,notes,label,label_signals,verdict,confidence,signal_types,format
30,2026-06-11T05:19:42.981923+00:00,batch2,test_images/payslip1.jpeg,V1,gemini-2.5-flash,0.05,14.50,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",authentic,high,[],True
31,2026-06-11T05:19:57.928182+00:00,batch2,test_images/payslip10.jpeg,V1,gemini-2.5-flash,0.05,14.94,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True
32,2026-06-11T05:20:29.880469+00:00,batch2,test_images/payslip2.png,V1,gemini-2.5-flash,0.05,31.95,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'medium'...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",tampered,medium,"[font_weight_inconsistency, sharpness_inconsis...",False
33,2026-06-11T05:20:41.214401+00:00,batch2,test_images/payslip20.png,V1,gemini-2.5-flash,0.05,11.33,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,authentic,[],tampered,high,[font_weight_inconsistency],True
34,2026-06-11T05:21:01.187514+00:00,batch2,test_images/payslip3.png,V1,gemini-2.5-flash,0.05,19.97,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",tampered,high,"[font_weight_inconsistency, font_mismatch]",False
35,2026-06-11T05:21:12.680004+00:00,batch2,test_images/payslip30.png,V1,gemini-2.5-flash,0.05,11.49,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True
36,2026-06-11T05:21:27.553605+00:00,batch2,test_images/payslip1.jpeg,V1_split,gemini-2.5-flash,0.10,14.81,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'medium'...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",tampered,medium,[font_weight_inconsistency],True
37,2026-06-11T05:21:38.359619+00:00,batch2,test_images/payslip10.jpeg,V1_split,gemini-2.5-flash,0.10,10.80,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True
38,2026-06-11T05:21:51.724211+00:00,batch2,test_images/payslip2.png,V1_split,gemini-2.5-flash,0.10,13.36,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",tampered,high,"[font_weight_inconsistency, resolution_mismatc...",True
39,2026-06-11T05:22:03.067231+00:00,batch2,test_images/payslip20.png,V1_split,gemini-2.5-flash,0.10,11.34,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,authentic,[],tampered,high,"[font_weight_inconsistency, resolution_mismatc...",True


#### If wanted to load only one batch for analysis

In [29]:
# current_batch = load_results(batch_id=BATCH_ID)
# current_batch[["image_path", "prompt_id", "verdict", "confidence", "signal_types", "format", "latency_s"]]